## Apresentação ✒️

Notebook destinado à realização do processo de avaliação das respostas dos modelos NLG sem a presença de ground truth. Tal cenário apresenta um particular desafio, tendo em vista que a presença de um texto de referência a partir do qual permite compreender a qualidade da resposta do modelo, com base na comparação, é o caminho clássico adotado e comumente recomendado durante o processo de avaliação de modelos de machine learning - especialmente num contexto de aprendizado supervisionado em que o ajuste do modelo de NLG a um determinado contexto pode ser enquadrado. 

Da mesma forma, apesar de se colocar como método padrão de análise, nem sempre a presença da ground truth pode ser verificada, quer seja motivada a não presença de um montante financeiro suficiente para pagar uma equipe de curadoria humana, quer seja por sua maior morosidade, aumentando o período de entrega de tais modelos em cenário produtivo. 

Nesse sentido, o que será coberto aqui será uma alternativa de processo de avaliação no qual não demanda a presença de uma ground truth para compreender a qualidade da resposta do modelo gerada e nem da base de conhecimento recuperada - pois o chatbot de demonstração se associa a um contexto de conversational RAG. Para tanto, será utilizado a mesma abordagem do notebook de `gen eval`, conhecida como LLM as a Judge, além das métricas BERT Score e Cossine Similarity, que oferecem quantitativamente a qualidade da resposta, segundo uma compreensão contextual fundamentada na distância entre os vetores de cada termo e da sentença como um todo, respectivamente.  

### Library 📓

In [1]:
import warnings
warnings.filterwarnings("ignore")

In [2]:
import os
import getpass
import evaluate
import numpy as np
import pandas as pd
import logging

import plotly.express as px
import plotly.graph_objects as go

from tqdm import tqdm

from typing import Dict, List

from IPython.display import Markdown

from scipy.stats import ttest_rel


from features.clean_memory import CleanMemory

from factor_analyzer.factor_analyzer import calculate_bartlett_sphericity

from prompts.system_message import system_message
from prompts.check_context import check_context_prompt
from prompts.contextualize_message import contextualize_prompt

from pandas import DataFrame

from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains.retrieval import create_retrieval_chain

from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_community.document_loaders import PyPDFLoader

from langchain_core.chat_history import (BaseChatMessageHistory,
                                         InMemoryChatMessageHistory)
from langchain_core.language_models import BaseChatModel
from langchain_core.messages import HumanMessage, trim_messages
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import (ChatPromptTemplate, MessagesPlaceholder,
                                    PromptTemplate)
from langchain_core.runnables import Runnable, RunnableBranch, RunnableLambda
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.output_parsers import PydanticOutputParser, JsonOutputParser

from langchain_core.vectorstores import InMemoryVectorStore, VectorStore

from langchain_groq import ChatGroq

from langchain_huggingface.embeddings import HuggingFaceEmbeddings

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_google_genai import ChatGoogleGenerativeAI

from pydantic import BaseModel, Field

from sentence_transformers import SentenceTransformer
from sentence_transformers.util import pairwise_cos_sim

### Carregando o dataset

In [9]:
file_name = "dataset_response_model.xlsx"

df = pd.read_excel(f"./data/{file_name}", engine="openpyxl")

In [ ]:
df = df.drop(["Base de conhecimento", "Ground Truth", "Unnamed: 0"], axis=1)

In [14]:
df.head()

,Question,Response Model,Kbs Recovered
0,O que são ghouls em Tokyo Ghoul e como eles se...,Os ghouls são criaturas muito semelhantes aos ...,['inhas entre \nbem e mal tornam-se tênues. \n...
1,Como e por que foi criada a organização CCG?,A organização CCG (Comissão de Contra-Ghoul) f...,['ca por um meio-\ntermo entre a sobrevivência...
2,O que acontece com Ken Kaneki após o transplan...,"Ken Kaneki, um estudante universitário, sofre ...","['nas do mangá, \ninfluenciando tendências est..."
3,Quais diferentes visões de convivência entre g...,"Em Tokyo Ghoul, existem diferentes visões de c...",['entre humanos e ghouls. Alguns \ngrupos de g...
4,: Qual é o significado de “One-Eyed King” no u...,"No universo de Tokyo Ghoul, o ""One-Eyed King"" ...",['inhas entre \nbem e mal tornam-se tênues. \n...


In [15]:
df.shape

(30, 3)

### Inicializando o modelo de LLM

In [ ]:
# api : your-api-key

os.environ["GROQ_API_KEY"]=getpass.getpass("Your API Key: ")

In [4]:
qwen_qwen  = "qwen/qwen3-32b"
mini_llama = "llama3-8b-8192"
llama_2    = "llama3-70b-8192"
llama      = "llama-3.3-70b-versatile"
deepseek   = "deepseek-r1-distill-llama-70b"

llm = ChatGroq(
    model = llama_2, 
    temperature = 0
)   

llm.invoke("Olá, tudo bem ?").content

'Olá! Tudo bem, obrigado! E você?'

### Embedding

In [5]:
%%time

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2"
)

CPU times: total: 1.02 s
Wall time: 3.15 s


### Formando a base de conhecimento 

A base de conhecimento utilizada se refere à lore do anime/ mangá Tokyo Ghoul - uma das melhores obras góticas - a partir da qual o modelo deverá utilizar para responder a certas perguntas do usuário, também sobre o tema. 

In [6]:
%%time

"""
Elaborando os métodos utilizados para o modelo possuir
a sua base de conhecimento.  
"""

loader = PyPDFLoader("./data/Tokyo Ghoul Knowledge Base.pdf").load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size         = 500, 
    chunk_overlap      = 50, 
    length_function    = len,
    separators         = ["", " ", ".", "\n", "\n\n"],
    is_separator_regex = False
).split_documents(loader)    

retriever = InMemoryVectorStore.from_documents( 
    documents = text_splitter,
    embedding = embeddings
).as_retriever(search_kwargs={"k": 2})

CPU times: total: 29.9 s
Wall time: 10.6 s


### Cossine Similarity Lib

Método responsável por promover uma análise objetiva - e quantitativa - da qualidade da resposta do modelo. Isso é feito segundo a compreensão que os termos presentes em linguagem natural podem ser compreendidos como vetores presentes num espaço dimensional no qual podem ser posicionados e mensurados a distância entre si, na forma em que quanto mais próximos um dos outros vetorialmente, tem-se que - tudo o mais constante - estarão também semanticamente. 

In [27]:
def sentence_embedding_similarity(
        dataset: DataFrame,
        column_user_message: str, 
        column_response_model: str, 
        index: int
    ) -> str:
    """ 
    Computes the cosine similarity between embeddings of 'ground truth' and 'response model' 
    from a specified row in the DataFrame.

    This function uses a pre-trained SentenceTransformer model to generate embeddings 
    for the specified 'response' and 'ground truth' texts in the DataFrame. The embeddings are 
    compared using cosine similarity to measure their semantic similarity.

    Args:
        dataset (DataFrame): A pandas DataFrame containing 'prompt' and 'response' columns.
        column_user_message: The name of user message's column.
        column_response_model: The name of response model's column.
        index (int): The row index in the DataFrame from which to extract the texts.

    Returns:
        float: The cosine similarity score between the embeddings of 'ground truth' and 'response model'.
    """
    model = SentenceTransformer("all-MiniLM-L6-v2")

    user_message = dataset.iloc[index][column_user_message]
    response_model = dataset.iloc[index][column_response_model]
    
    user_message_embedding = model.encode(user_message)
    response_embedding = model.encode(response_model)

    # Como a biblioteca SentenceTransformers espera vetores em 2D, 
    # tive que adicionar mais uma dimensão a cada embedding, formando
    # os respectivos expand embeddings a seguir, tanto para o prompt
    # quanto para a resposta gerada. 

    expand_prompt_embedding = np.expand_dims(user_message_embedding, axis=0)
    expand_response_embedding = np.expand_dims(response_embedding, axis=0)

    cossine_similarity = pairwise_cos_sim(
        expand_prompt_embedding, 
        expand_response_embedding
    )

    cossine_similarity_value = round(cossine_similarity[0].item(), 1)
    return cossine_similarity_value

### Avaliação - BERT Score

In [17]:
model = "distilbert-base-uncased"

In [ ]:
# Interagindo com o modelo: 

# Diferente da utilização padrão em que a avaliação é feita em conjunto 
# com a ground truth, aqui essa se dá em direação à mensagem enviada pelo usuário. 

bert_score = evaluate.load("bertscore")

bert_score_eval = bert_score.compute(
    predictions = [df.loc[5, "Kbs Recovered"]],
    references  = [df.loc[5, "Question"]], 
    model_type  = model  
)["f1"] 

f1_score = round(bert_score_eval[0], 3)
print(f"F1-score: {f1_score}")

F1-score: 0.711


#### Iterando com o modelo sobre o dataset

Iteração responsável pela formação da métrica que considera a corretude da resposta e da base de conhecimento recuperada, em função da base de conhecimento recuperada. Para reiteirar, concebe-se que quanto mais próximo de 1 for o valor encontrado, mais próximo vetorialmente está os termos e, portanto, são semanticamente mais próximos. 

In [20]:
f1_score_response = []
f1_score_kb_recovered = []

In [21]:
kb_column = "Kbs Recovered"
response_column = "Response Model"
user_message = "Question"

In [23]:
%%time

# Iteração que realiza a avaliação utilizando o BERT Score. 
# Para avaliar considerando a resposta do modelo ou o Kb recuperado
# em relação à mensagem do usuário, basta descomentar no código e informar
# o nome da coluna de interesse. 

for _, row in tqdm(df.iterrows(), desc="Avaliando com BERT Score:", total=df.shape[0]):

    bert_score_eval = bert_score.compute(
        predictions = [ row[f"{kb_column}"] ],
        references  = [ row[f"{user_message}"] ],
        model_type  = model
    )["f1"]

    f1_score = round(bert_score_eval[0], 1)
    # f1_score_response.append(f1_score)
    f1_score_kb_recovered.append(f1_score)


Avaliando com BERT Score::   0%|          | 0/30 [00:00<?, ?it/s]

Avaliando com BERT Score:: 100%|██████████| 30/30 [00:13<00:00,  2.26it/s]

CPU times: total: 47.8 s
Wall time: 13.3 s


In [24]:
# Adicionando as métricas geradas ao dataset, lembrando que o 
# valor encontrado se refere a média harmônica. 

df["bert_score_response"] = f1_score_response
df["bert_score_kb"] = f1_score_kb_recovered

### Avaliação - Cossine Similarity 

In [29]:
cossine_similarity = sentence_embedding_similarity(
    dataset               = df, 
    column_user_message   = "Question",
    column_response_model = "Response Model", 
    index                 = 5
)

print(f"Cossine Similarity: {cossine_similarity}")

Cossine Similarity: 0.5


In [30]:
cossine_similarity_response = []
cossine_similarity_kb = []

In [37]:
%%time

# Iteração que realiza a avaliação utilizando o Cossine Similarity. 
# Para avaliar considerando a resposta do modelo ou o Kb recuperado
# em relação à mensagem do usuário, basta descomentar no código e informar
# o nome da coluna de interesse. 

for i in tqdm(range(30), desc="Avaliando com C. Smilarity:"):

    cossine_similarity = sentence_embedding_similarity(
    dataset               = df, 
    column_user_message   = "Question",
    column_response_model = "Response Model", 
    index                 = i
    )

    cossine_similarity_response.append(cossine_similarity)
    # cossine_similarity_kb.append(cossine_similarity)

Avaliando com C. Smilarity:: 100%|██████████| 30/30 [00:09<00:00,  3.22it/s]

CPU times: total: 26.9 s
Wall time: 9.33 s


In [ ]:
# Adicionando as métricas geradas ao dataset :

df["cossine similarity response"] = cossine_similarity_response
df["cossine similarity kb"] = cossine_similarity_kb

### Avaliação - LLM as a Judge

In [77]:
judge_prompt = """\
  <role>
  Aja como um especialista em avaliação das respostas de chatbots conversacionais, 
  especializado em julgar a qualidade da resposta do modelo e da base de conhecimento recuperada
  em relação à mensagem do usuário, com base numa análise que considera premissa e hipótese e que 
  considera os seguintes <criterios>.
  </role>
  
  <criterios>
  Essas são as métricas com as quais você deve considerar para a análise da 
  resposta do modelo generativo e da base de conhecimento recuperada. 
  <corretude>: Avalia o quão correta está a <response_model> com base na <ground_truth>.
  <completude>: Avalia o quão completa está a <response_model> em relação ao <kb>.
  <resposta_embasada>: Avalia o quão embasada está a <response_model> em relação <kb>
  </criterios>

  <exemplos>
  Exemplos que ilustram como você deve avaliar a resposta do modelo generativo. 
  **Pergunta**: `Quem é o One-Eyed King em Tokyo Ghoul e qual é o seu papel na história?` **Resposta do modelo**: `O One-Eyed King é uma figura simbólica e messiânica no universo de Tokyo Ghoul. Inicialmente envolto em mistério, esse título representa a esperança de que ghouls e humanos possam coexistir. A identidade do One-Eyed King é revelada como sendo Ken Kaneki, que assume esse papel com o objetivo de unir facções opostas e pôr fim ao ciclo de violência. Sua posição desafia tanto os ghouls radicais quanto o CCG, tornando-o uma peça-chave na transformação do equilíbrio de poder.` **Avaliação**: Corretude: A resposta fornecida responde plenamente a mensagem do usuário - 1.0. Completude: A resposta cobre completamente a base de conhecimento fornecida. -1. Resposta embasada: A resposta do modelo está aderente à base de conhecimento fornecida - 1.
  </exemplos>
  
  <variaveis>
  <user_message>: {user_message}
  <response_model>: {response_model}
  <kb_recovered>: {kb_recovered}
  </variaveis>
  
  <avaliacao>
  Avalie a <resposta_do_modelo>, considerando à <pergunta> fornecida, com base em <criterios>.
  A sua avaliação conta com um conjunto de rúbrica (valor) e justificativa. A rúbrica varia segundo um intervalo de 0 a 1, sendo 1 a máxima pontuação e 0 a mínima. Pontuação de 0.8 indica acerto parcial, com falha em pelo menos uma das métricas e 0.4 um erro parcial, com pelo menos um acerto nas métricas. 
  Além da pontuação, forneça uma justificativa que fundamentou a sua pontuação, com uma explicação detalhada acerca dela, estrutura em formato de premissa e conclusão.
  Para a sua avaliação, considere as métricas fornecidas em <criterios> e avalie a resposta do modelo para cada uma das métricas.
  </avaliacao>
  
  <resposta>
  Responda **somente** português.
  A sua resposta deve considerar as métricas de <corretude>, <completude> e <resposta_embasada>.
  Formate a sua resposta utilizando o seguinte template: {format_instructions}
  </resposta>
"""

In [78]:
""" 
Criando a formatação da resposta esperada pelo judge. 
"""

class JudgeEval(BaseModel):
    criterio: str = Field(description="Critério avaliado")
    rubrica: float = Field(description="Valor da métrica avaliada")
    justificativa: str = Field(description="Justificativa da avaliação")

class JudgeOutput(BaseModel):
    avaliacoes: List[JudgeEval]

parser = PydanticOutputParser(pydantic_object=JudgeOutput)

In [79]:
def judge(
        user_message: str, 
        response_model: str, 
        kb_recovered: str,  
        llm = llm, 
        parser = parser
    ) -> str:
    """
    Evaluates the quality of a generative model's response using a language model (LLM) and predefined criteria.

    This function builds a prompt based on a question, the model's response, and the ground truth answer. It uses
    a chain-of-thought evaluation strategy with a language model to assess the response against four criteria:
    correctness, completeness, relevance, and overall performance.

    Args:
        question (str): The original user question that was asked.
        response_model (str): The response generated by the model being evaluated.
        ground_truth (str): The reference answer considered to be correct.
        llm: The language model used for generating the evaluation (default: global `llm`).
        parser: The parser used to structure and validate the LLM output (default: global `parser`).

    Returns:
        dict: A dictionary containing the evaluation results with metrics including rubric (score) and justification
              for each criterion: correctness, completeness, relevance, and overall performance.
    """ 
    judge_prompt_template = PromptTemplate(
        template          = judge_prompt, 
        input_variables   = ["response_model", 
                             "user_message", 
                             "kb_recovered"],
        partial_variables = {"format_instructions": parser.get_format_instructions()}
    )

    judge_chain = judge_prompt_template | llm | JsonOutputParser()

    try:
        judge_response = judge_chain.invoke(
            {
                "user_message": user_message, 
                "response_model": response_model,                
                "kb_recovered": kb_recovered
            }
        )
        return judge_response
    except Exception as e:
        print(f"[ERRO] Falha ao avaliar a linha com question: '{user_message[:30]}...'. Detalhes: {e}")
        return None

### Testando o Judge

In [88]:
user_message = df["Question"][15]
kb_recovered = df["Kbs Recovered"][15]
response_model = df["Response Model"][15]

In [89]:
print("Visualizando a pergunta, a resposta do modelo, a ground truth e a base de conhecimento recuperada:")

print(f"""
Pergunta: {user_message}\n
Response Model: {response_model}\n
Kb recuperado: {kb_recovered}
""")

Visualizando a pergunta, a resposta do modelo, a ground truth e a base de conhecimento recuperada:

Pergunta: De que forma a trilha sonora do anime contribui para a atmosfera de Tokyo Ghoul?

Response Model: A trilha sonora do anime Tokyo Ghoul contribui significativamente para a atmosfera sombria e dramática da série. As composições musicais, que mesclam coros sombrios e batidas eletrônicas, reforçam o tom melancólico e dramático da história, criando uma sensação de tensão e suspense. Além disso, a trilha sonora ajuda a realçar a atmosfera de horror e suspense, tornando a experiência de assistir ao anime mais imersiva e emocionante.

Kb recuperado: ['esafia a anatomia humana. Essa escolha artística ressalta \na alteridade dos ghouls e reforça o horror visual, ao mesmo tempo que torna cada \npersonagem mais identificável pelo tipo de kagune que manifesta. As artes originais do \nmangá exploram texturas macabras e sombreamento intenso, criando imagens icônicas \nque se espalharam pela c

In [90]:
%%time

"""
Testando o judge formato para um conjunto de texto 
abitrariamente escolhidos. 
"""

judge_response = judge(
    user_message   = user_message, 
    response_model = response_model, 
    kb_recovered   = kb_recovered
)

CPU times: total: 31.2 ms
Wall time: 1.82 s


In [91]:
judge_response["avaliacoes"]

[{'criterio': 'Corretude',
  'rubrica': 1.0,
  'justificativa': 'A resposta do modelo está plenamente alinhada com a pergunta do usuário, fornecendo uma explicação clara e completa sobre como a trilha sonora do anime Tokyo Ghoul contribui para a atmosfera da série.'},
 {'criterio': 'Completude',
  'rubrica': 0.8,
  'justificativa': 'A resposta do modelo cobre a maior parte da base de conhecimento recuperada, mas não menciona explicitamente a relação entre a trilha sonora e a atmosfera de horror e suspense. No entanto, a resposta fornece uma visão geral completa da contribuição da trilha sonora para a atmosfera da série.'},
 {'criterio': 'Resposta Embasada',
  'rubrica': 1.0,
  'justificativa': 'A resposta do modelo está fortemente embasada na base de conhecimento recuperada, utilizando conceitos e informações específicas sobre a trilha sonora do anime Tokyo Ghoul para sustentar sua explicação.'}]

### Iterando com as informações para o Judge

In [92]:
# Criando as colunas no dataset para cada uma das métricas.

for criterio in ['corretude', 'completude', 'resposta embasada']:
    df[f'{criterio}_rubrica'] = None
    df[f'{criterio}_justificativa'] = None

In [93]:
for i in tqdm(range(30), desc="Gerando a avaliação"):
    
    user_message = df["Question"][i]
    response_model = df["Response Model"][i] 
    kb_recovered = df["Kbs Recovered"][i]
    
    # Chama o judge
    result = judge(
    user_message   = user_message, 
    response_model = response_model, 
    kb_recovered   = kb_recovered
    )

    # Lista de avaliações
    if result is None:
        continue 
    
    metric_values = []
    avaliacoes = result["avaliacoes"]  

    for avaliacao in avaliacoes:
        criterio = avaliacao["criterio"].lower()  
        rubrica = avaliacao["rubrica"]
        justificativa = avaliacao["justificativa"]

        df.at[i, f"{criterio}_rubrica"] = rubrica
        df.at[i, f"{criterio}_justificativa"] = justificativa

        metric_values.append(rubrica)

Gerando a avaliação: 100%|██████████| 30/30 [09:21<00:00, 18.71s/it]


In [94]:
file_name = "dataset_without_ground_truth"
df.to_excel(f"{file_name}.xlsx")

### Próximos passos...

A partir desse processo, os próximos passos seriam semelhantes àqueles adotados no notebook de gen eval no qual há a verificação da correlação entre as métricas utilizadas, teste de hipótese realizado e também cobertura da qualidade de performance do modelo utilizado, considerando tanto a sua resposta quanto base de conhecimento recuperada. Não obstante, pode se dar o processo de realizar o monitoramento de data drifit. Para tanto a análise poderia se dirigir à compreensão dos tipos das mensagens enviadas pelo modelo e verificar se numa quantidade significativa destoa da distribuição na qual o modelo foi elaborado, tanto via Judge quanto BERT Score e Cossine Similarity. O mesmo pode se dar para as respostas do modelo, mas em cenário produtivo um indício disso pode vir de mensagens como de `I don't know`, precisamente acerca de sua prevelância nos logs obtidos. 